In [ ]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
topic_dir = None
for directory in (start, *start.parents):
    if (directory / "find_stack_files.py").is_file() and (directory / "config.yaml").is_file():
        topic_dir = directory
        break
if topic_dir is None:
    raise FileNotFoundError(
        "Topic folder not found. Open the notebook from topics/11-B-FLOODED-FOREST "
        "or set the kernel working directory there."
    )
if str(topic_dir) not in sys.path:
    sys.path.insert(0, str(topic_dir))

import find_stack_files as fs

fs.init_settings()

CREDENTIALS_PATH = fs.CREDENTIALS_PATH
OUTPUT_JSON_PATH = fs.OUTPUT_JSON_PATH
load_credentials = fs.load_credentials
get_maap_access_token = fs.get_maap_access_token
main = fs.main

print("Topic dir:", topic_dir)
print("Credentials:", CREDENTIALS_PATH)
print("Output JSON:", OUTPUT_JSON_PATH)


In [ ]:
# Query MAAP STAC for all ROIs in rois.json (uses paths from config.yaml)
main()


In [ ]:
import json
import urllib.request
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

json_path = Path(OUTPUT_JSON_PATH)
if not json_path.is_file():
    raise FileNotFoundError(
        f"{json_path} not found. Run the STAC search cell above first (main())."
    )

with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

offline_token, client_id, client_secret = load_credentials(CREDENTIALS_PATH)
access_token = get_maap_access_token(offline_token, client_id, client_secret)

print(f"Loaded: {json_path.resolve()}")
print(f"Collection: {data.get('collection')}")
print("Products per ROI:")
for roi in data.get("rois", []):
    print(f"  {roi['name']}: {roi['product_count']}")


In [ ]:
# Region to plot (must match a name printed in the cell above)
ROI_NAME = "Belém"

QL_ASSET_KEY = "enclosure_ql_pauli_png"  # quicklook_jpg, enclosure_coh_ql_png, ...
SCENE_FIG_BASE_INCHES = 12  # longer side of each per-scene figure
PRODUCT_SLICE = slice(None)  # e.g. slice(0, 3) to test; slice(None) = all products

roi_names = [r["name"] for r in data["rois"]]
if ROI_NAME not in roi_names:
    raise ValueError(f"Unknown ROI_NAME={ROI_NAME!r}. Choose from: {roi_names}")

roi = next(r for r in data["rois"] if r["name"] == ROI_NAME)
products = sorted(roi["products"], key=lambda p: p.get("datetime") or "")
print(f"{ROI_NAME}: {len(products)} products")
if products:
    print(f"  {products[0]['datetime']} … {products[-1]['datetime']}")


In [ ]:
from matplotlib.patches import Rectangle


def fetch_maap_image(url: str, token: str) -> Image.Image:
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
    with urllib.request.urlopen(req, timeout=120) as response:
        return Image.open(BytesIO(response.read()))


def scene_figsize(bbox, base_inches: float):
    """Figure size (width, height) from geographic bbox aspect ratio."""
    w, s, e, n = bbox
    lon_span = max(float(e - w), 1e-6)
    lat_span = max(float(n - s), 1e-6)
    ratio = lon_span / lat_span
    max_ratio = 4.0
    ratio = min(max(ratio, 1 / max_ratio), max_ratio)
    if ratio >= 1:
        return (base_inches * ratio, base_inches)
    return (base_inches, base_inches / ratio)


def add_roi_patch(ax, roi_bbox, **kwargs):
    defaults = {"fill": False, "edgecolor": "crimson", "linewidth": 2, "label": "ROI"}
    defaults.update(kwargs)
    ax.add_patch(
        Rectangle(
            (roi_bbox["west"], roi_bbox["south"]),
            roi_bbox["east"] - roi_bbox["west"],
            roi_bbox["north"] - roi_bbox["south"],
            **defaults,
        )
    )


to_show = products[PRODUCT_SLICE]
if not to_show:
    print("No products for this ROI — pick another ROI_NAME or re-run the STAC search.")
else:
    roi_bb = roi["bbox"]

    # Overview: all scene footprints + ROI
    fig, ax = plt.subplots(figsize=(9, 9))
    add_roi_patch(ax, roi_bb)
    for p in to_show:
        w, s, e, n = p["bbox"]
        ax.add_patch(Rectangle((w, s), e - w, n - s, fill=False, edgecolor="steelblue", alpha=0.5))
    pad = 0.05
    xs = [roi_bb["west"], roi_bb["east"]] + [p["bbox"][0] for p in to_show] + [p["bbox"][2] for p in to_show]
    ys = [roi_bb["south"], roi_bb["north"]] + [p["bbox"][1] for p in to_show] + [p["bbox"][3] for p in to_show]
    ax.set_xlim(min(xs) - pad, max(xs) + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + pad)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    ax.set_title(f"{ROI_NAME}: ROI vs {len(to_show)} scene extents")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    # One large figure per scene; quicklook stretched to STAC bbox (full swath extent)
    print(f"Plotting {len(to_show)} scenes (scroll notebook for each figure)...")
    for index, product in enumerate(to_show, start=1):
        url = product["assets"].get(QL_ASSET_KEY)
        w, s, e, n = product["bbox"]
        figsize = scene_figsize(product["bbox"], SCENE_FIG_BASE_INCHES)
        fig, ax = plt.subplots(figsize=figsize)

        if not url:
            ax.set_title(f"[{index}/{len(to_show)}] missing {QL_ASSET_KEY}")
            ax.axis("off")
            plt.show()
            continue

        img = fetch_maap_image(url, access_token)
        # Map PNG pixels to product geographic bbox (preview alignment; not survey-grade)
        ax.imshow(img, extent=(w, e, s, n), origin="upper", interpolation="nearest")
        add_roi_patch(ax, roi_bb)
        ax.set_xlim(w, e)
        ax.set_ylim(s, n)
        ax.set_xlabel("longitude")
        ax.set_ylabel("latitude")
        ax.set_aspect("equal", adjustable="box")
        short_id = product["id"].split("_")[-1] if "_" in product["id"] else product["id"][:24]
        ax.set_title(
            f"[{index}/{len(to_show)}] {product['datetime'][:19]}Z | "
            f"ROI overlap {product['roi_overlap_fraction']:.1%} | {short_id}",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()
